# Modül 09: Transformer ve Attention
## Deep Learning Path
---
## İçindekiler
1. Öğrenme Hedefleri
2. Scaled Dot-Product Attention
3. Multi-Head Attention
4. Positional Encoding
5. Transformer Encoder Bloğu
6. BERT vs GPT
7. PyTorch Mini Transformer
8. Attention Görselleştirme
9. Alıştırmalar

## 1. Öğrenme Hedefleri
- Scaled dot-product attention'ı sıfırdan uygulamak
- Multi-head attention'ın neden birden fazla kafa kullandığını açıklamak
- Positional encoding'i sinüzoidal formülle hesaplamak
- Transformer encoder bloğunu (MHA + FFN + AddNorm) uygulamak
- BERT ve GPT farkını açıklamak
- PyTorch ile mini transformer kodlamak

## 2. Scaled Dot-Product Attention

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

- **Q (Query)**: "Ne arıyorum?"
- **K (Key)**: "Neyle eşleşirim?"
- **V (Value)**: "Eşleşince ne döneceğim?"
- **$\sqrt{d_k}$**: Büyük boyutlarda dot product patlar → ölçekleme şart


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    dk = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(dk)
    if mask is not None: scores = scores + mask
    w = softmax(scores, axis=-1)
    return w @ V, w

# Demo
np.random.seed(1)
seq_len, dk = 5, 8
Q=np.random.randn(seq_len,dk); K=np.random.randn(seq_len,dk); V=np.random.randn(seq_len,dk)
out, attn_w = attention(Q, K, V)
print(f"Q,K,V: {Q.shape}  Çıkış: {out.shape}  Ağırlıklar: {attn_w.shape}")
print(f"Her satır toplamı 1: {np.allclose(attn_w.sum(1),1)}")
print(f"Satır 0 ağırlıkları: {attn_w[0].round(3)}")

# Causal (decoder) mask
mask = np.triu(np.full((seq_len,seq_len),-1e9), k=1)
_, causal_w = attention(Q, K, V, mask)
print(f"Causal mask - üst üçgen sıfır mı? {(causal_w[:,:]==(causal_w*np.tril(np.ones((5,5)))).round(6)).all()}")


## 3. Multi-Head Attention

$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1,...,\text{head}_h) \cdot W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

Her kafa farklı ilişkileri öğrenir: sözdizimi, anlam, konum...  
$d_k = d_{model} / h$ → boyut değişmez


In [ ]:
class MultiHeadAttention:
    def __init__(self, d_model, n_heads):
        self.h=n_heads; self.dk=d_model//n_heads; s=0.1
        self.WQ=np.random.randn(d_model,d_model)*s
        self.WK=np.random.randn(d_model,d_model)*s
        self.WV=np.random.randn(d_model,d_model)*s
        self.WO=np.random.randn(d_model,d_model)*s
    def split(self,x):
        sl=x.shape[0]; return x.reshape(sl,self.h,self.dk).transpose(1,0,2)
    def forward(self,Q,K,V):
        Qp=Q@self.WQ; Kp=K@self.WK; Vp=V@self.WV
        Qh=self.split(Qp); Kh=self.split(Kp); Vh=self.split(Vp)
        heads=[]; attns=[]
        for i in range(self.h):
            o,a=attention(Qh[i],Kh[i],Vh[i]); heads.append(o); attns.append(a)
        return np.concatenate(heads,axis=-1)@self.WO, np.array(attns)

d_model,n_heads,seq_len=16,4,6
mha=MultiHeadAttention(d_model,n_heads)
X=np.random.randn(seq_len,d_model)
out,attns=mha.forward(X,X,X)
print(f"d_model={d_model}, n_heads={n_heads}, d_k={d_model//n_heads}")
print(f"Giriş: {X.shape}  Çıkış: {out.shape}  Ağırlıklar: {attns.shape}")

# Görsel
fig,axes=plt.subplots(1,4,figsize=(13,3))
for i,ax in enumerate(axes):
    im=ax.imshow(attns[i],cmap='Blues',vmin=0,vmax=1)
    ax.set_title(f'Kafa {i+1}',fontweight='bold'); ax.axis('off')
plt.suptitle('Multi-Head Attention — 4 Kafa',fontweight='bold')
plt.tight_layout(); plt.show()


## 4. Positional Encoding

$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Eklenmiş token: $\text{embed}(x) + PE$


In [ ]:
def positional_encoding(seq_len, d_model):
    PE = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len).reshape(-1,1)
    i = np.arange(0, d_model, 2)
    div = np.exp(i * (-np.log(10000.0) / d_model))
    PE[:,0::2] = np.sin(pos * div)
    PE[:,1::2] = np.cos(pos * div)
    return PE

PE = positional_encoding(50, 64)
fig,ax = plt.subplots(figsize=(10,3.5))
im = ax.imshow(PE.T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
ax.set_title('Sinüzoidal Positional Encoding (50 pos, d=64)', fontweight='bold')
ax.set_xlabel('Pozisyon'); ax.set_ylabel('Encoding boyutu')
plt.colorbar(im, ax=ax, fraction=0.02)
plt.tight_layout(); plt.show()
print(f"PE değer aralığı: [{PE.min():.3f}, {PE.max():.3f}]")


## 5. Transformer Encoder Bloğu

```
x → [MHA Self-Attention] → Add & Norm → [FFN] → Add & Norm → çıkış
```

FFN: `max(0, xW₁+b₁)W₂+b₂` (d_ff = 4 × d_model tipik)

In [ ]:
class LayerNorm:
    def __init__(self,d,eps=1e-6): self.g=np.ones(d); self.b=np.zeros(d); self.eps=eps
    def forward(self,x): mu=x.mean(-1,keepdims=True); s=x.std(-1,keepdims=True); return self.g*(x-mu)/(s+self.eps)+self.b

class FFN:
    def __init__(self,d,dff): self.W1=np.random.randn(d,dff)*.1; self.b1=np.zeros(dff); self.W2=np.random.randn(dff,d)*.1; self.b2=np.zeros(d)
    def forward(self,x): return np.maximum(0,x@self.W1+self.b1)@self.W2+self.b2

class EncoderBlock:
    def __init__(self,d,h,dff):
        self.mha=MultiHeadAttention(d,h); self.ffn=FFN(d,dff)
        self.ln1=LayerNorm(d); self.ln2=LayerNorm(d)
    def forward(self,x):
        a,w=self.mha.forward(x,x,x); x=self.ln1.forward(x+a)
        f=self.ffn.forward(x); x=self.ln2.forward(x+f)
        return x,w

enc=EncoderBlock(16,4,32)
X=np.random.randn(6,16)
out,w=enc.forward(X)
print(f"Giriş: {X.shape}  Çıkış: {out.shape}")
print("✓ Boyut değişmedi! Transformer özelliği.")

# 4 blok üst üste
blocks=[EncoderBlock(16,4,32) for _ in range(4)]
x=X.copy()
for blk in blocks: x,_=blk.forward(x)
print(f"4 Blok sonrası: {x.shape}  ✓")


## 6. BERT vs GPT

| | BERT | GPT |
|--|------|-----|
| Mimari | Encoder | Decoder |
| Yön | Çift yönlü | Tek yönlü |
| Pre-training | Masked LM | Causal LM |
| Hedef | Anlama | Üretme |
| Örnek | BERT-base, RoBERTa | GPT-2/3/4 |

**BERT**: "The cat [MASK] on the mat" → predict "sat"  
**GPT**: "The cat sat" → predict next "on"


## 7. Alıştırmalar

**1.** `attention()` fonksiyonuna batch boyutu ekle: `(batch, seq, d_k)`.

**2.** Decoder için causal mask'ı otomatik oluşturan fonksiyon yaz.

**3.** `EncoderBlock` sınıfına dropout ekle (training/inference modlu).

**4.** PyTorch'ta 2 encoder + 1 decoder transformer yaz, seq2seq görevi için.

**5.** `positional_encoding` ile iki farklı pozisyonun PE farkının ne kadar lineer olduğunu göster.

---
**Sonraki Modül:** Modül 10 — GAN ve VAE
